[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/monai_dataloading_bottleneck.ipynb)

# Find where a MONAI training run waits

MONAI's `SupervisedTrainer` runs its data loop inside the engine, so a 3D segmentation job can look busy while the GPU waits for patches. This notebook answers a practical question:

> Which of MONAI's own data-loading settings removes that wait, and where does the bottleneck go once it is gone?

You will train the same 3D UNet on the spleen task of the Medical Segmentation Decathlon six times. Each run changes one setting from the run before it. TraceML's `TraceMLHandler` diagnoses every run, and `traceml compare` shows what each change did.

The result depends on the hardware. The numbers quoted below come from one Tesla T4 with four CPU cores, where the notebook took 21 minutes once the data was downloaded. A two-core Colab runtime spends longer on every CPU-side step, so it takes longer and its verdicts can differ.

**Before you start:** switch to a GPU runtime via *Runtime -> Change runtime type -> Hardware accelerator: GPU*. TraceML is open source: `pip install traceml-ai` ([github.com/traceopt-ai/traceml](https://github.com/traceopt-ai/traceml)).

## The comparison

| Run | Change from the run before | Flags after `--args --data-dir data` |
|---|---|---|
| 1 | Baseline: `Dataset` with `DataLoader(num_workers=0)` | none |
| 2 | `num_workers=4` | `--num-workers 4` |
| 3 | `CacheDataset(cache_rate=1.0)` | `--dataset cache --num-workers 4` |
| 4 | `num_workers=0` | `--dataset cache` |
| 5 | `ThreadDataLoader(num_workers=0)` | `--dataset cache --loader thread` |
| 6 | `SupervisedTrainer(amp=True)`, a compute setting | `--dataset cache --loader thread --amp` |

Run 4 isolates the worker count after caching. MONAI suggests `ThreadDataLoader` for light preprocessing on cached data, because it avoids passing batches between worker processes. Without run 4, run 5 would change the loader and the worker count at once.

The model, transforms, patch sampler, batch size, seed and step count stay fixed. Every step trains on 8 patches of 96x96x96 voxels: 2 volumes with 4 patches each. Each run is 5 epochs of 16 steps, 80 steps in all. Mixed precision stays off until run 6, so runs 1 to 5 compare data-loading settings only.

TraceML records the loop's Input Wait, forward, backward and optimizer time for every step, plus the host-to-device copy (H2D) on a GPU. Input Wait comes from the engine's own batch-fetch events. For `ThreadDataLoader` that is the time the training loop waited, not the time the background thread spent preparing the batch.

## Choose a run mode

`extended` is the default GPU demonstration on the spleen data. `smoke` is a small CPU-only verification mode for contributors and CI: it writes a few synthetic volumes to a temporary directory, sends them through the same transforms, patch sampler, `SupervisedTrainer` and `TraceMLHandler`, and downloads nothing. It checks that the summary counts four steps and reports Input Wait, forward, backward and optimizer time.

In [ ]:
import os

# Change the default to "smoke" to run the small CPU-only path manually.
RUN_MODE = os.environ.get("TRACEML_NOTEBOOK_MODE", "extended")
if RUN_MODE not in {"extended", "smoke"}:
    raise ValueError("TRACEML_NOTEBOOK_MODE must be 'extended' or 'smoke'")
print(f"TraceML notebook mode: {RUN_MODE}")

## 1. Check the GPU (extended mode only)

In [ ]:
import torch

if RUN_MODE == "extended":
    !nvidia-smi -L
    print("CUDA available:", torch.cuda.is_available())
    print("CPU cores:", os.cpu_count())
    assert (
        torch.cuda.is_available()
    ), "No GPU. Runtime -> Change runtime type -> GPU, then rerun."
else:
    print("Smoke mode uses CPU; no GPU is required.")

## 2. Install TraceML with the MONAI extra

The `monai` extra installs MONAI and PyTorch Ignite next to the PyTorch build Colab already provides. `nibabel` reads the NIfTI volumes. Outside Colab, install the extras shown in the comment below.

In [ ]:
import os

if os.environ.get("TRACEML_NOTEBOOK_SKIP_INSTALL") == "1":
    print("Notebook smoke runner provided TraceML dependencies.")
else:
    %pip install -q "traceml-ai[monai]" nibabel
# Outside Colab or in a fresh CPU environment: %pip install -q "traceml-ai[torch,monai]" nibabel

## 3. The complete training script

A standard MONAI `SupervisedTrainer` on the preprocessing from MONAI's spleen tutorial. The TraceML additions are the two lines marked in the script: `traceml_monai.init()` once, and `TraceMLHandler()` in `train_handlers`. Each flag changes one setting, and `--smoke` runs the CPU verification path.

The script prints one `[record]` line per run. It holds the versions and the fixed settings. The dataset, loader, worker and AMP settings are read back from the trainer, so the record shows what ran. It also holds two timings that no TraceML step covers. `dataset_ready_s` is the time to build the dataset before training, and `train_s` is the wall time of `trainer.run()`. Section 7 reads it.

In [ ]:
%%writefile monai_dataloading_bottleneck.py
"""Find where a MONAI training run waits: spleen segmentation.

Trains a 3D UNet on patches from the Medical Segmentation Decathlon spleen
task (Task09_Spleen, CC BY-SA 4.0) under one MONAI ``SupervisedTrainer``.
Each flag changes one setting, so two runs that differ in one flag measure
that setting alone:

    --dataset plain|cache    Dataset, or CacheDataset(cache_rate=1.0)
    --num-workers N          loader worker processes
    --loader torch|thread    DataLoader, or ThreadDataLoader
    --amp                    SupervisedTrainer(amp=True)

Launch through ``traceml run`` (a bare ``python`` run trains untraced):

    traceml run --mode summary --logs-dir logs --run-name spleen_1_baseline \\
        monai_dataloading_bottleneck.py --args --data-dir data
    traceml run --mode summary --logs-dir logs --run-name spleen_2_workers \\
        monai_dataloading_bottleneck.py --args --data-dir data \\
        --num-workers 4
    traceml compare logs/spleen_1_baseline/final_summary.json \\
        logs/spleen_2_workers/final_summary.json

CPU-only check without the dataset (what the notebook smoke job runs):

    traceml run --mode summary --logs-dir logs --run-name monai_smoke \\
        monai_dataloading_bottleneck.py --args --smoke --epochs 2

The spleen archive is downloaded into ``--data-dir`` on first use.
``--smoke`` writes small synthetic volumes to a temporary directory and
sends them through the same transforms and patch sampler. Reading NIfTI
files needs ``nibabel``.
"""

from __future__ import annotations

import argparse
import contextlib
import glob
import json
import os
import subprocess
import sys
import tempfile
import time

import ignite
import monai
import numpy as np
import torch
from monai.apps import download_and_extract
from monai.data import CacheDataset, DataLoader, Dataset, ThreadDataLoader
from monai.engines import SupervisedTrainer
from monai.losses import DiceLoss
from monai.networks.layers import Norm
from monai.networks.nets import UNet
from monai.transforms import (
    Compose,
    CropForegroundd,
    EnsureChannelFirstd,
    LoadImaged,
    Orientationd,
    RandCropByPosNegLabeld,
    ScaleIntensityRanged,
    Spacingd,
)
from monai.utils import set_determinism

import traceml_ai
from traceml_ai.integrations import monai as traceml_monai

SEED = 42
SPLEEN_URL = (
    "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar"
)
SPLEEN_MD5 = "410d4a301da4e5b2f6f86ec3ddba524e"
# The task has 41 labelled volumes. MONAI's spleen tutorial keeps the last
# nine for validation; this study trains on the same 32 and runs no
# validation.
LABELLED = 41
HELD_OUT = 9
PATCH = (96, 96, 96)
PATCHES_PER_VOLUME = 4
CACHE_RATE = 1.0
# Fixed, so the cache is built the same way whatever the loader settings.
CACHE_WORKERS = 4
SMOKE_PATCH = (32, 32, 32)
SMOKE_SHAPE = (64, 64, 40)
SMOKE_VOLUMES = 4


def spleen_files(data_dir):
    """Download Task09_Spleen once and return the 32 training pairs."""
    root = os.path.join(data_dir, "Task09_Spleen")
    if not os.path.isdir(root):
        download_and_extract(
            SPLEEN_URL,
            os.path.join(data_dir, "Task09_Spleen.tar"),
            data_dir,
            hash_val=SPLEEN_MD5,
        )
    images = sorted(glob.glob(os.path.join(root, "imagesTr", "*.nii.gz")))
    labels = sorted(glob.glob(os.path.join(root, "labelsTr", "*.nii.gz")))
    # An interrupted extraction leaves a partial directory, which would
    # otherwise train on fewer volumes without saying so.
    if len(images) != LABELLED or len(labels) != LABELLED:
        raise RuntimeError(
            f"Expected {LABELLED} labelled volumes under {root}, found "
            f"{len(images)} images and {len(labels)} labels. Delete the "
            "directory and run again to download a fresh copy."
        )
    pairs = [{"image": i, "label": s} for i, s in zip(images, labels)]
    return pairs[:-HELD_OUT]


def write_synthetic_volumes(directory, count=SMOKE_VOLUMES):
    """Write small CT-like volumes, each with one bright labelled organ."""
    import nibabel as nib

    rng = np.random.default_rng(SEED)
    affine = np.diag([0.8, 0.8, 2.5, 1.0])
    grid = np.indices(SMOKE_SHAPE)
    pairs = []
    for index in range(count):
        centre = [rng.integers(12, size - 12) for size in SMOKE_SHAPE]
        distance = sum((g - c) ** 2 for g, c in zip(grid, centre))
        organ = distance <= 8**2
        image = rng.normal(0.0, 30.0, SMOKE_SHAPE) + 100.0 * organ
        pair = {}
        for key, array in (
            ("image", image.astype(np.float32)),
            ("label", organ.astype(np.uint8)),
        ):
            pair[key] = os.path.join(directory, f"{key}_{index}.nii.gz")
            nib.save(nib.Nifti1Image(array, affine), pair[key])
        pairs.append(pair)
    return pairs


def train_transforms(patch):
    """
    MONAI's spleen tutorial preprocessing, then its random patch sampler.

    Everything before ``RandCropByPosNegLabeld`` is deterministic, so
    ``CacheDataset`` stores its output once and only the crop runs per
    iteration.
    """
    keys = ["image", "label"]
    return Compose(
        [
            LoadImaged(keys=keys),
            EnsureChannelFirstd(keys=keys),
            ScaleIntensityRanged(
                keys=["image"],
                a_min=-57,
                a_max=164,
                b_min=0.0,
                b_max=1.0,
                clip=True,
            ),
            CropForegroundd(keys=keys, source_key="image", allow_smaller=True),
            Orientationd(keys=keys, axcodes="RAS"),
            Spacingd(
                keys=keys,
                pixdim=(1.5, 1.5, 2.0),
                mode=("bilinear", "nearest"),
            ),
            RandCropByPosNegLabeld(
                keys=keys,
                label_key="label",
                spatial_size=patch,
                pos=1,
                neg=1,
                num_samples=PATCHES_PER_VOLUME,
                image_key="image",
                image_threshold=0,
            ),
        ]
    )


def build_dataset(kind, files, patch):
    transforms = train_transforms(patch)
    if kind == "cache":
        return CacheDataset(
            files,
            transforms,
            cache_rate=CACHE_RATE,
            num_workers=CACHE_WORKERS,
            progress=False,
        )
    return Dataset(files, transforms)


def build_loader(kind, dataset, batch_size, num_workers):
    # MONAI's loaders collate the sampler's patches into one batch, so a
    # batch holds batch_size * PATCHES_PER_VOLUME patches.
    loader = ThreadDataLoader if kind == "thread" else DataLoader
    return loader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )


def build_network():
    return UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
        norm=Norm.BATCH,
    )


def _gpu_and_driver(device):
    if device.type != "cuda":
        return None, None
    # The driver is host-wide, so any row answers. nvidia-smi numbers GPUs
    # physically, not the way CUDA_VISIBLE_DEVICES renumbers them.
    try:
        rows = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=driver_version",
                "--format=csv,noheader",
            ],
            capture_output=True,
            text=True,
            check=True,
            timeout=10,
        ).stdout.split()
    except (OSError, subprocess.SubprocessError):
        rows = []
    return torch.cuda.get_device_name(device), rows[0] if rows else None


def run_record(argv, trainer, device, patch, dataset_s, train_s):
    """
    Everything needed to repeat or compare this run, as one dict.

    The dataset, loader, worker and AMP settings are read back from the
    trainer and its loader, so the record shows what ran rather than what
    was asked for. The rest are the script's fixed settings.
    """
    gpu, driver = _gpu_and_driver(device)
    loader = trainer.data_loader
    cached = isinstance(loader.dataset, CacheDataset)
    return {
        "argv": list(argv),
        "gpu": gpu,
        "driver": driver,
        "torch": torch.__version__,
        "monai": monai.__version__,
        "ignite": ignite.__version__,
        "traceml": traceml_ai.__version__,
        "seed": SEED,
        "dataset": type(loader.dataset).__name__,
        "volumes": len(loader.dataset),
        "cache_rate": CACHE_RATE if cached else None,
        "cached_volumes": loader.dataset.cache_num if cached else None,
        "loader": type(loader).__name__,
        "num_workers": loader.num_workers,
        "amp": trainer.amp,
        "batch_size": loader.batch_size,
        "patches_per_volume": PATCHES_PER_VOLUME,
        "patch_size": list(patch),
        "epochs": trainer.state.max_epochs,
        "steps": trainer.state.iteration,
        "dataset_ready_s": round(dataset_s, 3),
        "train_s": round(train_s, 3),
    }


def build_parser():
    parser = argparse.ArgumentParser(
        description=(
            "3D UNet on MONAI's spleen task under SupervisedTrainer; each "
            "flag changes one setting."
        ),
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--dataset",
        choices=["plain", "cache"],
        default="plain",
        help="Dataset, or CacheDataset holding every preprocessed volume.",
    )
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument(
        "--loader",
        choices=["torch", "thread"],
        default="torch",
        help="monai.data.DataLoader, or ThreadDataLoader.",
    )
    parser.add_argument(
        "--amp",
        action="store_true",
        help="Mixed precision in SupervisedTrainer, a compute setting.",
    )
    parser.add_argument(
        "--data-dir",
        default="data",
        help="Where the spleen archive is downloaded and extracted.",
    )
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument(
        "--smoke",
        action="store_true",
        help="CPU-only synthetic volumes; downloads nothing.",
    )
    return parser


def main(argv=None):
    argv = sys.argv[1:] if argv is None else argv
    args = build_parser().parse_args(argv)
    set_determinism(seed=SEED)

    # TraceML line 1: arms H2D timing. Input Wait comes from the engine's
    # own batch-fetch events, so it is right for ThreadDataLoader too.
    traceml_monai.init()

    cuda = torch.cuda.is_available() and not args.smoke
    device = torch.device("cuda" if cuda else "cpu")
    patch = SMOKE_PATCH if args.smoke else PATCH
    print(
        f"[demo] dataset={args.dataset} num_workers={args.num_workers} "
        f"loader={args.loader} amp={args.amp} smoke={args.smoke} "
        f"device={device} batch={args.batch_size} patch={patch} "
        f"epochs={args.epochs}",
        flush=True,
    )

    with contextlib.ExitStack() as stack:
        if args.smoke:
            scratch = stack.enter_context(tempfile.TemporaryDirectory())
            files = write_synthetic_volumes(scratch)
        else:
            files = spleen_files(args.data_dir)

        # CacheDataset preprocesses every volume here, before training, so
        # none of this time is inside a TraceML step.
        start = time.perf_counter()
        dataset = build_dataset(args.dataset, files, patch)
        dataset_s = time.perf_counter() - start
        print(f"[demo] dataset ready in {dataset_s:.1f} s", flush=True)

        network = build_network().to(device)
        trainer = SupervisedTrainer(
            device=device,
            max_epochs=args.epochs,
            train_data_loader=build_loader(
                args.loader, dataset, args.batch_size, args.num_workers
            ),
            network=network,
            optimizer=torch.optim.Adam(network.parameters(), lr=1e-4),
            loss_function=DiceLoss(to_onehot_y=True, softmax=True),
            amp=args.amp,
            train_handlers=[traceml_monai.TraceMLHandler()],  # TraceML line 2
        )
        start = time.perf_counter()
        trainer.run()
        if device.type == "cuda":
            torch.cuda.synchronize()
        train_s = time.perf_counter() - start

    record = run_record(argv, trainer, device, patch, dataset_s, train_s)
    print("[record] " + json.dumps(record, sort_keys=True), flush=True)
    return record


if __name__ == "__main__":
    main()


## 4. Get the data (extended mode only)

The spleen task of the Medical Segmentation Decathlon has 41 labelled CT volumes. This study trains on the first 32 after sorting, as MONAI's spleen tutorial does, and runs no validation. The archive is 1.6 GB. It comes from the mirror MONAI's own tutorials use, and the script checks its MD5 before extracting it.

**Licence.** The Medical Segmentation Decathlon data is distributed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/); the spleen volumes come from Memorial Sloan Kettering Cancer Center. Cite Antonelli et al., "The Medical Segmentation Decathlon", *Nature Communications* 13, 4128 (2022), [doi:10.1038/s41467-022-30695-9](https://doi.org/10.1038/s41467-022-30695-9). The notebook downloads the data when it runs; the TraceML repository does not redistribute it.

In [ ]:
if RUN_MODE == "extended":
    from monai_dataloading_bottleneck import spleen_files

    files = spleen_files("data")
    print("training volumes:", len(files))
else:
    print("Smoke mode writes its own synthetic volumes.")

## 5. Run the six configurations

Always launch through `traceml run`: it starts the aggregator that receives the handler's measurements and writes the summary. A bare `python` run trains untraced.

### Run 1: the baseline

A plain `Dataset` with `DataLoader(num_workers=0)`. Before every step the training process loads two CT volumes, resamples them and crops the patches, while the GPU waits. Smoke mode runs four CPU-only steps on synthetic volumes instead.

On the T4 this run was `INPUT-BOUND`: Input Wait 5,654.6 ms of a 6,469.6 ms step (87%), compute 770.5 ms.

In [ ]:
if RUN_MODE == "smoke":
    !traceml run --mode summary --logs-dir logs --run-name monai_smoke monai_dataloading_bottleneck.py --args --smoke --epochs 2
else:
    !traceml run --mode summary --logs-dir logs --run-name spleen_1_baseline monai_dataloading_bottleneck.py --args --data-dir data

### Run 2: four worker processes (extended mode only)

Same dataset, now with `num_workers=4`, so four processes load and resample volumes ahead of the training loop.

On the T4 this run was `INPUT-BOUND`: Input Wait 2,489.8 ms of a 3,225.3 ms step (77%), compute 701.9 ms.

In [ ]:
if RUN_MODE == "extended":
    !traceml run --mode summary --logs-dir logs --run-name spleen_2_workers monai_dataloading_bottleneck.py --args --data-dir data --num-workers 4
else:
    print("Smoke mode runs one configuration, not the comparison.")

### Run 3: cache the preprocessed volumes (extended mode only)

`CacheDataset(cache_rate=1.0)` runs every transform before the random crop once, before training, and keeps the result in memory. Only the crop runs per step. Building the cache is not inside any TraceML step, so the script times it separately: it took 45.3 s here.

On the T4 this run was `INPUT-BOUND`: Input Wait 101.6 ms of a 830.6 ms step (12%), compute 706.0 ms.

In [ ]:
if RUN_MODE == "extended":
    !traceml run --mode summary --logs-dir logs --run-name spleen_3_cache monai_dataloading_bottleneck.py --args --data-dir data --dataset cache --num-workers 4
else:
    print("Smoke mode runs one configuration, not the comparison.")

### Run 4: cached, no workers (extended mode only)

The cached dataset from run 3, back to `num_workers=0`. The random crop now runs in the training process.

On the T4 this run was `INPUT-BOUND`: Input Wait 195.4 ms of a 917.2 ms step (21%), compute 703.5 ms.

In [ ]:
if RUN_MODE == "extended":
    !traceml run --mode summary --logs-dir logs --run-name spleen_4_cache_no_workers monai_dataloading_bottleneck.py --args --data-dir data --dataset cache
else:
    print("Smoke mode runs one configuration, not the comparison.")

### Run 5: ThreadDataLoader (extended mode only)

`ThreadDataLoader` prepares the next batch on a background thread while the current step trains, still with no worker processes.

On the T4 this run was `COMPUTE-BOUND`: Input Wait 38.2 ms of a 765.6 ms step (5%), compute 706.8 ms.

In [ ]:
if RUN_MODE == "extended":
    !traceml run --mode summary --logs-dir logs --run-name spleen_5_thread monai_dataloading_bottleneck.py --args --data-dir data --dataset cache --loader thread
else:
    print("Smoke mode runs one configuration, not the comparison.")

### Run 6: mixed precision (extended mode only)

`SupervisedTrainer(amp=True)` changes compute, not data loading. It shows what happens to the verdict once the step itself gets faster.

On the T4 this run was `INPUT-BOUND`: Input Wait 76.5 ms of a 228.7 ms step (33%), compute 115.4 ms.

In [ ]:
if RUN_MODE == "extended":
    !traceml run --mode summary --logs-dir logs --run-name spleen_6_amp monai_dataloading_bottleneck.py --args --data-dir data --dataset cache --loader thread --amp
else:
    print("Smoke mode runs one configuration, not the comparison.")

## 6. Compare each pair

`traceml compare` prints a compact comparison of two summaries and writes JSON and text artifacts for each pair. Each pair differs in one setting. Step Time here is on the GPU clock.

On one T4 with four CPU cores the five comparisons read:

| Pair | Change | Step Time | Input Wait | Compute | Diagnosis | `compare` verdict |
|---|---|---|---|---|---|---|
| 1 to 2 | `num_workers=4` | 6,469.6 ms to 3,225.3 ms (-50.1%) | 5,654.6 ms to 2,489.8 ms (-56.0%) | 770.5 ms to 701.9 ms (-8.9%) | INPUT-BOUND to INPUT-BOUND | IMPROVEMENT |
| 2 to 3 | `CacheDataset` | 3,225.3 ms to 830.6 ms (-74.2%) | 2,489.8 ms to 101.6 ms (-95.9%) | 701.9 ms to 706.0 ms (+0.6%) | INPUT-BOUND to INPUT-BOUND | IMPROVEMENT |
| 3 to 4 | `num_workers=0` | 830.6 ms to 917.2 ms (+10.4%) | 101.6 ms to 195.4 ms (+92.3%) | 706.0 ms to 703.5 ms (-0.4%) | INPUT-BOUND to INPUT-BOUND | REGRESSION |
| 4 to 5 | `ThreadDataLoader` | 917.2 ms to 765.6 ms (-16.5%) | 195.4 ms to 38.2 ms (-80.5%) | 703.5 ms to 706.8 ms (+0.5%) | INPUT-BOUND to COMPUTE-BOUND | IMPROVEMENT |
| 5 to 6 | `amp=True` | 765.6 ms to 228.7 ms (-70.1%) | 38.2 ms to 76.5 ms (+100.4%) | 706.8 ms to 115.4 ms (-83.7%) | COMPUTE-BOUND to INPUT-BOUND | IMPROVEMENT |

Four workers halve the step, but the loop still waits for most of it. Each worker loads and resamples whole CT volumes, and four of them do not keep up with the GPU.

Caching removes the loading and resampling from every step. Step Time falls a further 74.2%, and GPU utilization rises from 21.6% to 89.8%.

Run 4 is the control for run 5. Without workers the crop runs in the training process, and Step Time rises 10.4%. `ThreadDataLoader` moves the crop to a background thread: Input Wait falls 80.5%, the diagnosis turns `COMPUTE-BOUND`, and the GPU is 96.9% busy.

Mixed precision cuts compute by 83.7% and Step Time by 70.1%. The diagnosis goes back to `INPUT-BOUND`, because Input Wait rises from 38.2 ms to 76.5 ms. In run 4 the crop cost a median 196.7 ms per step. The thread can hide that behind 706.8 ms of compute, but not behind 115.4 ms.

In [ ]:
import json
from pathlib import Path

if RUN_MODE == "smoke":
    summary_dir = Path("logs/monai_smoke")
    for name in ("final_summary.json", "final_summary.txt"):
        assert (summary_dir / name).is_file(), f"Missing {summary_dir / name}"
    summary = json.loads((summary_dir / "final_summary.json").read_text())
    steps = summary["step_time"]["metadata"]["training_total_steps"]
    assert steps == 4, f"Expected 4 smoke steps, got {steps}"
    average = summary["step_time"]["global"]["average"]
    for phase in (
        "input_wait_ms",
        "forward_ms",
        "backward_ms",
        "optimizer_ms",
    ):
        # Absent means never measured; a measured region cannot take zero.
        assert average.get(phase) is not None, f"no {phase}: {average}"
        assert average[phase] > 0, f"zero {phase}: {average}"
    print(f"Smoke check passed: {steps} steps; artifacts are in {summary_dir}")
else:
    !traceml compare logs/spleen_1_baseline/final_summary.json logs/spleen_2_workers/final_summary.json --output=logs/compare_1_2
    !traceml compare logs/spleen_2_workers/final_summary.json logs/spleen_3_cache/final_summary.json --output=logs/compare_2_3
    !traceml compare logs/spleen_3_cache/final_summary.json logs/spleen_4_cache_no_workers/final_summary.json --output=logs/compare_3_4
    !traceml compare logs/spleen_4_cache_no_workers/final_summary.json logs/spleen_5_thread/final_summary.json --output=logs/compare_4_5
    !traceml compare logs/spleen_5_thread/final_summary.json logs/spleen_6_amp/final_summary.json --output=logs/compare_5_6

## 7. All six runs side by side

The cell below prints one row per run from each run's summary and `[record]` line. On the T4 it printed:

| Run | Verdict | Steps | Step Time | Input Wait | Compute | H2D | GPU util | Dataset ready | `trainer.run()` |
|---|---|---|---|---|---|---|---|---|---|
| 1 | INPUT-BOUND | 80 | 6,469.6 ms | 5,654.6 ms | 770.5 ms | 24.7 ms | 11.3% | 0.0 s | 517.9 s |
| 2 | INPUT-BOUND | 80 | 3,225.3 ms | 2,489.8 ms | 701.9 ms | 48.3 ms | 21.6% | 0.0 s | 259.5 s |
| 3 | INPUT-BOUND | 80 | 830.6 ms | 101.6 ms | 706.0 ms | 32.9 ms | 89.8% | 45.3 s | 67.5 s |
| 4 | INPUT-BOUND | 80 | 917.2 ms | 195.4 ms | 703.5 ms | 24.7 ms | 75.6% | 45.8 s | 73.7 s |
| 5 | COMPUTE-BOUND | 80 | 765.6 ms | 38.2 ms | 706.8 ms | 27.8 ms | 96.9% | 45.5 s | 61.4 s |
| 6 | INPUT-BOUND | 80 | 228.7 ms | 76.5 ms | 115.4 ms | 28.0 ms | 51.4% | 45.6 s | 18.5 s |

TraceML's numbers agree with the wall clock. Summed over the 80 steps, its mean Step Time accounts for 98.4% to 99.9% of the measured `trainer.run()` time in every run.

Caching moves 45.3 to 45.8 s of preprocessing out of the steps and in front of training, where no TraceML step sees it. For 80 steps that is a large share of the run. Run 5 took 45.5 s to build its cache and 61.4 s to train. Run 2 took 259.5 s to train. A longer run builds the cache once and saves time on every step.

Every run used a Tesla T4 (driver 595.71.05), torch 2.11.0+cu130, MONAI 1.6.0, Ignite 0.5.5, TraceML 0.4.1.dev7+g2da37940c.d20260922 and seed 42. Each run's `[record]` line in `logs/<run>/nodes/node_0/training.stdout.log` holds its exact arguments and the settings read back from the trainer.

In [ ]:
import json
import re
from pathlib import Path

RUNS = (
    ["monai_smoke"]
    if RUN_MODE == "smoke"
    else [
        "spleen_1_baseline",
        "spleen_2_workers",
        "spleen_3_cache",
        "spleen_4_cache_no_workers",
        "spleen_5_thread",
        "spleen_6_amp",
    ]
)


def record(run):
    # The script prints its settings and timings as one [record] line.
    log = Path("logs") / run / "nodes" / "node_0" / "training.stdout.log"
    line = re.search(r"^\[record\] (.+)$", log.read_text(), re.M)
    return json.loads(line.group(1))


def load(run):
    # The summary holds TraceML's measurements.
    root = Path("logs") / run
    summary = json.loads((root / "final_summary.json").read_text())
    average = summary["step_time"]["global"]["average"]
    gpu = summary["system"]["global"]["average"].get("gpu_util_percent")
    return {
        "run": run,
        "verdict": summary["primary_diagnosis"]["status"],
        "steps": summary["step_time"]["metadata"]["training_total_steps"],
        "step ms": average["step_time_ms"],
        "input ms": average["input_wait_ms"],
        "compute ms": average["compute_ms"],
        "H2D ms": average.get("h2d_ms"),
        "GPU %": gpu,
        "dataset s": record(run)["dataset_ready_s"],
        "train s": record(run)["train_s"],
    }


rows = [load(run) for run in RUNS]
columns = list(rows[0])
widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in columns}
widths.update({c: max(len(c), 9) for c in columns[3:]})
print("  ".join(f"{c:>{widths[c]}}" for c in columns))
for row in rows:
    out = []
    for c in columns:
        value = row[c]
        if isinstance(value, float):
            value = f"{value:.1f}"
        out.append(f"{'n/a' if value is None else value:>{widths[c]}}")
    print("  ".join(out))

## 8. What the run average hides: the first step of each epoch

The summary averages over all 80 steps. TraceML also keeps each step's phases in the run's telemetry database next to the summary, within its history retention window (30 minutes by default; `traceml run --history-retention` changes it). The cell below reads the per-step Input Wait, which is the CPU-side time of each batch fetch, and compares the median step with the first step of each epoch.

On the T4, run 3's median step waited 27.6 ms. Each of its 5 epoch-start steps waited 807.2 ms to 1,519.1 ms, and those 5 steps hold 74.4% of the run's Input Wait. The four worker processes start again at every epoch, because `persistent_workers` is off by default, so the first batch of each epoch waits for them. `persistent_workers=True` would be the next setting to test; this notebook does not measure it.

Run 2 waits at every epoch start too, 11.18 s to 16.14 s each. Most of its wait falls inside epochs, though: 19 of its 75 other steps waited more than a second. Run 5's thread starts with an empty buffer at each epoch, so its epoch-start steps wait 153.4 ms to 273.9 ms, about one crop.

Run 6 has no such concentration. Its median step waited 73.8 ms, and its epoch-start steps hold only 19.0% of its wait. That pattern matches the crop no longer fitting inside the shorter step, not a start-up cost.

In [ ]:
import json
import sqlite3
import statistics
from pathlib import Path

PREFIX = "_traceml_internal:"


def wait_per_step(run):
    # One row per step for rank 0; a repeated run under the same name appends
    # rows, so keep the latest row per step. The fetch is a CPU wait, so read
    # its CPU clock.
    db = Path("logs") / run / "aggregator" / "telemetry"
    query = (
        "SELECT step, events_json FROM step_time_samples WHERE id IN ("
        "  SELECT MAX(id) FROM step_time_samples"
        "  WHERE global_rank = 0 GROUP BY step) ORDER BY step"
    )
    waits = {}
    with sqlite3.connect(db) as conn:
        for step, events_json in conn.execute(query):
            fetch = json.loads(events_json).get(PREFIX + "dataloader_next", {})
            waits[step] = sum(d.get("cpu_ms") or 0.0 for d in fetch.values())
    assert waits, f"No step rows in {db}; was history retention disabled?"
    return waits


print(f"{'run':>26}  {'median':>9}  {'epoch starts':>24}  {'share':>6}")
for run in RUNS:
    rec = record(run)
    per_epoch = max(1, rec["steps"] // rec["epochs"])
    waits = wait_per_step(run)
    starts = [w for s, w in waits.items() if (s - 1) % per_epoch == 0]
    total = sum(waits.values())
    share = f"{100 * sum(starts) / total:5.1f}%" if total else "   n/a"
    low, high = min(starts), max(starts)
    print(
        f"{run:>26}  {statistics.median(waits.values()):7.1f} ms"
        f"  {low:9.1f} to {high:7.1f} ms  {share}"
    )

## Use this in your own trainer

In an existing MONAI script, call `traceml_monai.init()` once and add `traceml_monai.TraceMLHandler()` to the `SupervisedTrainer`'s `train_handlers`. Launch it through `traceml run --mode summary ...`. Only the `SupervisedTrainer` is traced. An evaluator gets one warning and nothing attached, so validation does not count toward Input Wait.

If TraceML says `INPUT-BOUND`, change one setting at a time, in the order this notebook does. Keep a change only when wall time and Input Wait improve on the hardware that will run the job. After a compute change such as mixed precision, measure again: a faster step can expose input work that used to hide behind it.

Guide: [MONAI integration](https://github.com/traceopt-ai/traceml/blob/main/docs/user_guide/integrations/monai.md).